# RescueVision - E4 damage classifier (DINOv2)

In [ ]:
import subprocess, sys, os, json, time, traceback
from pathlib import Path

WORK = Path("/kaggle/working")
REPO = Path("/tmp/RescueVision")
if not REPO.exists():
    subprocess.run(["git", "clone", "-q", "https://github.com/AliOmarAbdelhady/RescueVision.git", str(REPO)], check=True)
sys.path.insert(0, str(REPO / "src"))

def mark(stage, ok=True, extra=""):
    """Progress ledger in /kaggle/working: survives kills where the log does not."""
    rec = {"t": time.strftime("%H:%M:%S"), "stage": stage, "ok": bool(ok), "extra": str(extra)[:1500]}
    with open(WORK / "_progress.txt", "a") as f:
        f.write(json.dumps(rec) + "\n")
    (WORK / "_status.json").write_text(json.dumps(rec, indent=1))
    print("[mark]", rec, flush=True)

os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
mark("setup:start")
# pin albumentations: newer versions (preinstalled on the GPU image) have a
# stricter shape check that rejects our verified-consistent (HxWx3, HxW) pairs;
# --upgrade forces the downgrade past pip's "already satisfied"; all transforms
# ALSO use is_check_shapes=False as a version-proof second layer
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--upgrade",
                "albumentations==2.0.8"], check=False)
import albumentations as _A
mark("setup:albumentations", extra=_A.__version__)
print("albumentations", _A.__version__)

import torch
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# num_workers=0: DataLoader workers died on the Kaggle GPU image; pin_memory
# off too (useless with workers=0, and a hard-kill vector on some drivers)
NW = 0
PIN = False
print("torch", torch.__version__, "| device:", DEVICE)
print(subprocess.run(["df", "-h", "/tmp", "/kaggle/working"], capture_output=True, text=True).stdout)
mark("setup:done", extra=str(DEVICE))

# per-epoch heartbeat: every trainer that logs an epoch also lands in the marks,
# so a silent kill still tells us how far training got and with what metrics
import rescuevision.training.logging_utils as _lu
_orig_log = _lu.CSVLogger.log
def _marked_log(self, metrics, epoch):
    try:
        mark(f"epoch{epoch}", extra=str({k: round(float(v), 4) for k, v in metrics.items()
                                         if isinstance(v, (int, float))})[:400])
    except Exception:
        pass
    return _orig_log(self, metrics, epoch)
_lu.CSVLogger.log = _marked_log
print("epoch heartbeat patch applied")

In [ ]:
import numpy as np
import torch
import albumentations as A
from albumentations.pytorch import ToTensorV2

from rescuevision.constants import IMAGENET_MEAN, IMAGENET_STD
from rescuevision.data import datasets as _D
from PIL import Image as _PIL
from pathlib import Path as _PPath

def _load_arr(path):
    """Read .png or .npy array transparently."""
    path = _PPath(path)
    if path.suffix == ".npy":
        return np.load(path)
    return np.array(_PIL.open(path))

def _find_tile(directory, tile_id):
    for ext in (".png", ".npy"):
        p = _PPath(directory) / f"{tile_id}{ext}"
        if p.exists():
            return p
    raise FileNotFoundError(f"{tile_id} in {directory}")

def joint_seg_transforms(cfg, phase="train", image_size=512):
    """Joint pipeline for a stacked pre+post pair (H, W, 6): one geometric draw for both."""
    mean, std = IMAGENET_MEAN * 2, IMAGENET_STD * 2
    tf = [A.Resize(image_size, image_size)]
    if phase == "train":
        if cfg.get("horizontal_flip", 0) > 0:
            tf.append(A.HorizontalFlip(p=cfg["horizontal_flip"]))
        if cfg.get("vertical_flip", 0) > 0:
            tf.append(A.VerticalFlip(p=cfg["vertical_flip"]))
        if cfg.get("rotate90", 0) > 0:
            tf.append(A.RandomRotate90(p=cfg["rotate90"]))
        if cfg.get("brightness_contrast", 0) > 0:
            tf.append(A.RandomBrightnessContrast(p=cfg["brightness_contrast"]))
    if cfg.get("normalize", True):
        tf.append(A.Normalize(mean=mean, std=std))
    tf.append(ToTensorV2())
    return A.Compose(tf, is_check_shapes=False)

def joint_clf_pair_transforms(cfg, phase="train", image_size=224):
    """Joint pipeline for stacked pre+post crops (H, W, 6)."""
    mean, std = IMAGENET_MEAN * 2, IMAGENET_STD * 2
    tf = [A.Resize(image_size, image_size)]
    if phase == "train":
        if cfg.get("horizontal_flip", 0) > 0:
            tf.append(A.HorizontalFlip(p=cfg["horizontal_flip"]))
        if cfg.get("vertical_flip", 0) > 0:
            tf.append(A.VerticalFlip(p=cfg["vertical_flip"]))
        if cfg.get("rotate", 0) > 0:
            tf.append(A.Rotate(limit=cfg["rotate"], p=0.5))
        if cfg.get("brightness_contrast", 0) > 0:
            tf.append(A.RandomBrightnessContrast(p=cfg["brightness_contrast"]))
    if cfg.get("normalize", True):
        tf.append(A.Normalize(mean=mean, std=std))
    tf.append(ToTensorV2())
    return A.Compose(tf, is_check_shapes=False)

def _nocheck(tf):
    # disable albumentations' shape pre-check: newer builds (preinstalled on the
    # Kaggle GPU image) reject our verified HxW-consistent image/mask pairs
    try:
        return A.Compose(tf.transforms, is_check_shapes=False)
    except Exception:
        return tf

def _joint_dmg_seg_getitem(self, idx):
    tile_id = self.tile_ids[idx]
    pre_image = _load_arr(_find_tile(self.pre_dir, tile_id))
    post_image = _load_arr(_find_tile(self.post_dir, tile_id))
    mask = _load_arr(_find_tile(self.masks_dir, tile_id))
    pair = np.concatenate([pre_image, post_image], axis=2)
    if self.transform:
        augmented = self.transform(image=pair, mask=mask)
        image, mask = augmented["image"], augmented["mask"]
    else:
        image = torch.from_numpy(pair.transpose(2, 0, 1))
    mask = mask.long() if torch.is_tensor(mask) else torch.from_numpy(mask).long()
    return {"image": image, "mask": mask, "tile_id": tile_id}

def _joint_change_getitem(self, idx):
    tile_id = self.tile_ids[idx]
    pre_image = _load_arr(_find_tile(self.pre_dir, tile_id))
    post_image = _load_arr(_find_tile(self.post_dir, tile_id))
    mask = _load_arr(_find_tile(self.masks_dir, tile_id))
    pair = np.concatenate([pre_image, post_image], axis=2)
    if self.transform:
        augmented = self.transform(image=pair, mask=mask)
        pair_t, mask = augmented["image"], augmented["mask"]
    else:
        pair_t = torch.from_numpy(pair.transpose(2, 0, 1))
    mask = mask.long() if torch.is_tensor(mask) else torch.from_numpy(mask).long()
    return {"pre_image": pair_t[:3], "post_image": pair_t[3:], "mask": mask, "tile_id": tile_id}

_D.XBDDamageSegmentationDataset.__getitem__ = _joint_dmg_seg_getitem
_D.XBDChangeDetectionDataset.__getitem__ = _joint_change_getitem

def _joint_crop_getitem(self, idx):
    row = self.crops_df.iloc[idx]
    pre_crop = _load_arr(row["pre_crop_path"])
    post_crop = _load_arr(row["post_crop_path"])
    label = self.label_map[row["label_id"]]
    pair = np.concatenate([pre_crop, post_crop], axis=2)
    if self.transform:
        pair_t = self.transform(image=pair)["image"]
    else:
        pair_t = torch.from_numpy(pair.transpose(2, 0, 1))
    return {"pre_crop": pair_t[:3], "post_crop": pair_t[3:], "label": label,
            "crop_id": row["crop_id"], "tile_id": row["tile_id"]}

_D.XBDDamageCropDataset.__getitem__ = _joint_crop_getitem

def _tile_dataset_init(self, processed_dir, split_csv, split="train", mask_type="building_binary",
                       transform=None, image_size=512, debug_limit=None):
    import pandas as pd
    self.processed_dir = _PPath(processed_dir)
    self.images_dir = self.processed_dir / "images" / "pre"
    self.pre_dir = self.processed_dir / "images" / "pre"
    self.post_dir = self.processed_dir / "images" / "post"
    if type(self).__name__ == "XBDDamageSegmentationDataset":
        self.masks_dir = self.processed_dir / "masks" / "damage_multiclass"
    elif type(self).__name__ == "XBDChangeDetectionDataset":
        self.masks_dir = self.processed_dir / "masks" / "change_binary"
    else:
        self.masks_dir = self.processed_dir / "masks" / mask_type
    self.transform = transform
    self.image_size = image_size
    splits_df = pd.read_csv(split_csv)
    tile_ids = splits_df[splits_df["split"] == split]["tile_id"].tolist()
    if debug_limit:
        tile_ids = tile_ids[:debug_limit]
    self.tile_ids = [t for t in tile_ids
                     if _find_tile(self.pre_dir, t).exists()
                     and _find_tile(self.post_dir, t).exists()
                     and _find_tile(self.masks_dir, t).exists()]
    print(f"{type(self).__name__}: split={split}, {len(self.tile_ids)} tiles")

def _xbd_seg_getitem(self, idx):
    tile_id = self.tile_ids[idx]
    image = _load_arr(_find_tile(self.pre_dir, tile_id))
    mask = _load_arr(_find_tile(self.masks_dir, tile_id))
    if self.transform:
        aug = self.transform(image=image, mask=mask)
        image, mask = aug["image"], aug["mask"]
    mask = mask.long() if torch.is_tensor(mask) else torch.from_numpy(mask).long()
    return {"image": image, "mask": mask, "tile_id": tile_id}

_D.XBDSegmentationDataset.__init__ = _tile_dataset_init
_D.XBDDamageSegmentationDataset.__init__ = _tile_dataset_init
_D.XBDChangeDetectionDataset.__init__ = _tile_dataset_init
_D.XBDSegmentationDataset.__getitem__ = _xbd_seg_getitem
mark("patch:done")
print("png-aware dataset init patch applied")
print("joint pre/post transform patch applied")

In [ ]:
import time
import pandas as pd
mark("tar:wait")
TAR = None
if (Path("/tmp/processed_xbd") / "metadata" / "splits_event_aware.csv").exists():
    print("processed tree already at /tmp/processed_xbd (smoke/debug path)")
else:
    for _attempt in range(40):   # up to 20 min for the kernel-source mount
        for _p in Path("/kaggle/input").rglob("processed_xbd.tar"):
            TAR = _p
            break
        if TAR is not None:
            break
        _ents = [p.name for p in Path("/kaggle/input").iterdir()] if Path("/kaggle/input").exists() else []
        print(f"[{_attempt+1}/40] waiting for tar mount, /kaggle/input: {_ents}", flush=True)
        time.sleep(30)
    assert TAR is not None, "processed_xbd.tar never mounted - kernel-source attach failed"
    _gib = TAR.stat().st_size / 1e9
    print("tar:", TAR, f"{_gib:.2f} GB")
    mark("tar:mounted", extra=f"{_gib:.2f}GB")
    subprocess.run(["tar", "-xf", str(TAR), "-C", "/tmp"], check=True)
PROCESSED = Path("/tmp/processed_xbd")
assert (PROCESSED / "metadata" / "splits_event_aware.csv").exists(), "extracted tree missing splits csv"
_splits = pd.read_csv(PROCESSED / "metadata" / "splits_event_aware.csv")
print("tiles per split:", _splits["split"].value_counts().to_dict())
mark("tar:extracted", extra=str(_splits["split"].value_counts().to_dict()))

In [ ]:
import logging
import pandas as pd
from torch.utils.data import DataLoader
from rescuevision.data.datasets import XBDDamageCropDataset
from rescuevision.models.dinov2_classifier import DINOv2Classifier
from rescuevision.models.losses import FocalLoss, compute_class_weights
from rescuevision.training.train_damage_classifier import train_damage_classifier
from rescuevision.utils.seed import set_seed

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
OUTPUT = WORK
name = "damage_classifier_dinov2"
METRIC = "macro_f1"

SEED = 42
AUG = {"horizontal_flip": 0.5, "vertical_flip": 0.5, "rotate": 15, "brightness_contrast": 0.3, "normalize": True}
BS = 32
MAX_CROPS = 12000     # quota-friendly cap, deterministic subsample
EPOCHS = 10           # stage 1: frozen head | stage 2: last 4 blocks unfrozen
UNFREEZE_AFTER = 6
PATIENCE = 3

config = {
    "seed": SEED, "device": DEVICE.type,
    "model": {"unfreeze_after_epoch": UNFREEZE_AFTER, "unfreeze_blocks": 4},
    "training": {"epochs": EPOCHS, "lr": 1e-3, "stage2_lr": 1e-5, "weight_decay": 1e-4,
                  "scheduler": "cosine", "mixed_precision": True,
                  "early_stopping_patience": PATIENCE, "save_best_metric": METRIC},
}

set_seed(SEED)
# rewrite relative crop paths from the processed tree to absolute
crops = pd.read_csv(PROCESSED / "metadata" / "buildings.csv")
splits = pd.read_csv(PROCESSED / "metadata" / "splits_event_aware.csv")
crops["pre_crop_path"] = str(PROCESSED) + "/" + crops["pre_crop_path"]
crops["post_crop_path"] = str(PROCESSED) + "/" + crops["post_crop_path"]
crops_csv = OUTPUT / "building_crops_mounted.csv"
crops.to_csv(crops_csv, index=False)
print(f"{len(crops)} classified crops total")

split_csv = PROCESSED / "metadata" / "splits_event_aware.csv"
train_ds = XBDDamageCropDataset(crops_csv, split_csv, split="train",
                                transform=joint_clf_pair_transforms(AUG, "train", 224))
val_ds = XBDDamageCropDataset(crops_csv, split_csv, split="val",
                              transform=joint_clf_pair_transforms(AUG, "val", 224))
test_ds = XBDDamageCropDataset(crops_csv, split_csv, split="test",
                               transform=joint_clf_pair_transforms(AUG, "val", 224))

# deterministic cap on train crops only (keep val/test intact)
if len(train_ds) > MAX_CROPS:
    rng = np.random.default_rng(SEED)
    keep = rng.permutation(len(train_ds))[:MAX_CROPS]
    train_ds.crops_df = train_ds.crops_df.iloc[sorted(keep)].reset_index(drop=True)
print(f"train={len(train_ds)} val={len(val_ds)} test={len(test_ds)}")
mark("dino:data", extra=f"train={len(train_ds)}")

train_loader = DataLoader(train_ds, batch_size=BS, shuffle=True, num_workers=NW, pin_memory=False, persistent_workers=bool(NW))
val_loader = DataLoader(val_ds, batch_size=BS, shuffle=False, num_workers=NW, pin_memory=False, persistent_workers=bool(NW))
test_loader = DataLoader(test_ds, batch_size=BS, shuffle=False, num_workers=NW, pin_memory=False)

label_names = ["no-damage", "minor-damage", "major-damage", "destroyed"]
train_labels = train_ds.crops_df["label_id"].map({1: 0, 2: 1, 3: 2, 4: 3}).tolist()
class_weights = compute_class_weights(train_labels, num_classes=4).to(DEVICE)
print("class weights:", [round(w, 3) for w in class_weights.tolist()])

model = DINOv2Classifier(backbone="facebook/dinov2-base", num_classes=4,
                         freeze_backbone=True, mlp_hidden_dim=512, dropout=0.3).to(DEVICE)
loss_fn = FocalLoss(gamma=2.0, alpha=class_weights)


try:
    _it = iter(train_loader)
    for _ in range(2):
        _b = next(_it)
        with torch.amp.autocast(DEVICE.type, enabled=True):
            _lg = model(_b["pre_crop"].to(DEVICE), _b["post_crop"].to(DEVICE))
            _loss = loss_fn(_lg, _b["label"].to(DEVICE))
        _loss.backward()
        model.zero_grad(set_to_none=True)
    if DEVICE.type == "cuda":
        torch.cuda.synchronize()
        mark("probe:ok", extra=f"peak={torch.cuda.max_memory_allocated()/1e9:.1f}GB")
    else:
        mark("probe:ok", extra="cpu")
except Exception:
    mark("probe:FAIL", extra=traceback.format_exc()[-1200:])
    raise
mark("dino:model-ready")
mark("dino:train-start", extra=f"train={len(train_ds)}")
best = train_damage_classifier(model, train_loader, val_loader, loss_fn, config,
                               output_dir=str(Path("/tmp/run") / name), class_names=label_names)
print("best:", best)
mark("dino:train-done", extra=str(best))

In [ ]:
import shutil as _shutil
_shutil.copytree(f"/tmp/run/damage_classifier_dinov2", WORK / "damage_classifier_dinov2", dirs_exist_ok=True)
print("results copied to", WORK / "damage_classifier_dinov2")
mark("copyout:damage_classifier_dinov2")
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

log_df = pd.read_csv(next((WORK / "damage_classifier_dinov2" / "logs").glob("*.csv")))
metric = "macro_f1"
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(log_df["epoch"], log_df["train_loss"], label="train")
if "val_loss" in log_df:
    axes[0].plot(log_df["epoch"], log_df["val_loss"], label="val")
axes[0].set_title("Loss"); axes[0].set_xlabel("epoch"); axes[0].legend(); axes[0].grid(alpha=0.3)
axes[1].plot(log_df["epoch"], log_df[metric], color="tab:green")
axes[1].set_title(f"val {metric} (best={log_df[metric].max():.4f} @ epoch {int(log_df.loc[log_df[metric].idxmax(), 'epoch'])})")
axes[1].set_xlabel("epoch"); axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.savefig(WORK / "damage_classifier_dinov2" / "training_curves.png", dpi=120, bbox_inches="tight")
plt.show()

In [ ]:
import json

ckpt = torch.load(WORK / name / "checkpoints" / "best.pt", map_location=DEVICE, weights_only=False)
model.load_state_dict(ckpt["model_state_dict"])
model.to(DEVICE).eval()
print(f"best checkpoint: epoch {ckpt['epoch']}, {METRIC}={ckpt.get(f'best_{METRIC}'):.4f}")

summary = {
    "run": name,
    "best_epoch": int(ckpt["epoch"]),
    f"best_val_{METRIC}": float(ckpt.get(f"best_{METRIC}")),
    "history_best": float(pd.read_csv(next((WORK / name / "logs").glob("*.csv")))[METRIC].max()),
}
with open(WORK / name / "summary.json", "w") as f:
    json.dump(summary, f, indent=2)
print(json.dumps(summary, indent=2))

import numpy as np
from rescuevision.training.train_damage_classifier import _evaluate_classifier
val_metrics = _evaluate_classifier(model, val_loader, DEVICE, loss_fn, True)
test_metrics = _evaluate_classifier(model, test_loader, DEVICE, loss_fn, True)
print("VAL :", {k: round(v, 4) for k, v in val_metrics.items() if isinstance(v, float)})
print("TEST:", {k: round(v, 4) for k, v in test_metrics.items() if isinstance(v, float)})

y_true, y_pred = [], []
model.eval()
with torch.no_grad():
    for batch in test_loader:
        with torch.amp.autocast(DEVICE.type, enabled=True):
            logits = model(batch["pre_crop"].to(DEVICE), batch["post_crop"].to(DEVICE))
        y_pred.extend(logits.argmax(1).cpu().numpy())
        y_true.extend(batch["label"].numpy())
from sklearn.metrics import classification_report, confusion_matrix
print(classification_report(y_true, y_pred, target_names=label_names, zero_division=0))
import seaborn as sns, matplotlib.pyplot as plt
cm = confusion_matrix(y_true, y_pred)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=label_names, yticklabels=label_names)
plt.title("DINOv2 classifier - test confusion"); plt.tight_layout()
plt.savefig(WORK / name / "test_confusion.png", dpi=120, bbox_inches="tight")
plt.show()
mark("dino:eval-done")